In [ ]:
import hist
from coffea.util import load, save
import mplhep as hep
import matplotlib.pyplot as plt
import copy
import numpy as np
import os
from tabulate import tabulate
import uncertainties as unc  
import uncertainties.unumpy as unumpy 
plt.style.use(hep.style.CMS)



In [ ]:
year = "2022post"
histversion = year #'2022pre'#22Test'#'GenIsoPhoTest' # 'sdfjTest'#
whathist = {
    '2022pre': '2022_0702',
    '2022post':'2022EE_0702',
    '2023pre': '2023_0702',
    '2023post': '2023BPix_0702'

}
lumi = {
    '2022pre' : 7.99,
    '2022post': 26.68,
    '2023pre': 17.96,
    '2023post': 9.68
}
if 'Test' in histversion:
    hists = load('./hadmonotop'+whathist[histversion]+'.scaled')
else:
    hists = load('./hadmonotop'+whathist[histversion]+'.scaled')
    #hists = load('./hadmonotop'+year+'_'+whathist[histversion]+'.scaled')
hists['sig']

In [ ]:
hists['bkg']

In [ ]:
## Merge VV
for dataset in ['bkg']:
    print(f'Now {dataset} processing')
    for key in hists[dataset].keys():
        hists[dataset][key]['VV'] = hists[dataset][key]['WW'] + hists[dataset][key]['WZ'] + hists[dataset][key]['ZZ']
        for p in ['WW', 'WZ', 'ZZ']:
            del hists[dataset][key][p]
        #print(f'{key} variable check: {hists[dataset][key].keys()}')

In [ ]:
stack2 = {
    'sr':[r'Z ($\nu\nu$) + Jets',r'W ($\ell\nu$) + Jets', 'WW', 'WZ', 'ZZ','TT','G + Jets', 'Single Top', 'Z ($\ell\ell$) + Jets', 'QCD Multijet' ],
    'tw':[r'W ($\ell\nu$) + Jets', 'TT','QCD Multijet', 'WW', 'WZ', 'ZZ','Single Top', 'Z ($\ell\ell$) + Jets',r'Z ($\nu\nu$) + Jets', 'G + Jets'],
    'zr':['Z ($\ell\ell$) + Jets',r'W ($\ell\nu$) + Jets', 'TT','QCD Multijet', 'WW', 'WZ', 'ZZ', 'Single Top',  r'Z ($\nu\nu$) + Jets', 'G + Jets'],
    'gr':['G + Jets',r'W ($\ell\nu$) + Jets', 'TT','QCD Multijet', 'WW', 'WZ', 'ZZ', 'Single Top', 'Z ($\ell\ell$) + Jets',r'Z ($\nu\nu$) + Jets'],
}
stack = {
    'sr':[r'Z ($\nu\nu$) + Jets',r'W ($\ell\nu$) + Jets', 'VV','TT','G + Jets','ST', 'Z ($\ell\ell$) + Jets', 'QCD Multijet' ],
    'tw':[r'W ($\ell\nu$) + Jets', 'TT','QCD Multijet', 'VV','ST', 'Z ($\ell\ell$) + Jets',r'Z ($\nu\nu$) + Jets', 'G + Jets'],
    'zr':['Z ($\ell\ell$) + Jets',r'W ($\ell\nu$) + Jets', 'TT','QCD Multijet', 'VV','ST',  r'Z ($\nu\nu$) + Jets', 'G + Jets'],
    'gr':['G + Jets',r'W ($\ell\nu$) + Jets', 'TT','QCD Multijet', 'VV', 'ST',  'Z ($\ell\ell$) + Jets',r'Z ($\nu\nu$) + Jets'],
}

cat_colors = {
    r'W ($\ell\nu$) + Jets': "#1f77b4", 
    'TT': "#ff7f0e", 
    'QCD Multijet': "#2ca02c",
    'VV': "#9467bd", 
#    'WZ': "#8c564b", 
#    'ZZ': "#e377c2",
    'ST': "#aec7e8", 
    r'Z ($\ell\ell$) + Jets': "#ffbb78", 
    r'Z ($\nu\nu$) + Jets': "#98df8a",
    'G + Jets': "#ff9896"
}


colors = {
    region: [cat_colors[proc] for proc in procs]
    for region, procs in stack.items()
}

regions = ['sr', 'wmcr', 'wecr', 'tmcr', 'gcr', 'tecr', 'zecr', 'zmcr']
leg_reg = {
    'sr': 'SR', 'wmcr': r'W($\mu$) CR', 'wecr': r'W(e) CR', 'tmcr': r'Top($\mu$) CR', 'gcr': 'G CR',
    'tecr': r'Top(e) CR', 'zecr':r'Z(ee) CR', 'zmcr': r'Z($\mu\mu$) CR'
}


In [ ]:
hists['bkg']['ut_bin2']['TT'][{'region': 'tecr','TvsQCD':0}]

In [ ]:
hists['bkg']['ut_bin2']['TT'][{'region': 'tecr','TvsQCD':sum}]

In [ ]:
######
## Run key loop: SR pass
######
import matplotlib.pyplot as plt
import numpy as np
import mplhep as hep
from cycler import cycler
plt.style.use(hep.style.CMS)

region = 'sr'
stacks = hists['bkg']['sumw'].keys()
stack_map = 'sr'

keys = ['fjeta','fjphi','fjpt','jpt','jeta','jphi','met','metphi','ut','ut_bin','ut_bin2','uphi',
        'mindphirecoil','minDphirecoil','njets','njbtagL','mupt', 'mueta', 'muphi'
       ]
sig_key1 = 'sig_Mphi-2000_Mchi-150'
sig_key2 = 'sig_Mphi-200_Mchi-150'
os.makedirs(f"plots{year}_{whathist[histversion]}", exist_ok=True)

for key in keys:
    fig, (ax, rax) = plt.subplots(
        nrows=2,
        ncols=1,
        figsize=(10,10),
        gridspec_kw={"height_ratios": (3, 1)},
        sharex=True
    )

    fig.subplots_adjust(hspace=.07)
    hep.cms.label(ax=ax, llabel='Private Work', rlabel=str(lumi[year])+' fb$^{-1}$ (13.6 TeV)')
    #ax.set_prop_cycle(cycler(color=colors[stack_map]))

    bkg  = hists['bkg']
    data = hists['data']
    sig  = hists['sig']

    mcstack = None
    
    for sample in stack[stack_map]:
        #if 'Z ($\nu\nu$)' in sample: continue
        #print(sample)
        try:
            target = bkg[key][sample][{'region': region, 'TvsQCD':sum}]
            bins = target.axes.edges[0]

        except:
            print(f'In {region} sample of {sample} is Not exist')
            continue

        try:
            mcstack += bkg[key][sample][{'region': region, 'TvsQCD':sum}]
            drawstack += bkg[key][sample][{'region': region, 'TvsQCD':sum}].values()
        except:
            print("Start adding the stack mc & data")
            mcstack = bkg[key][sample][{'region': region, 'TvsQCD':sum}]
            drawstack = bkg[key][sample][{'region': region, 'TvsQCD':sum}].values()

    for sample in stack[stack_map]:
        if region not in bkg[key][sample].axes["region"]:
            print(f"{region} is not in {sample}'s hist axis")
            continue
        ax.hist(bins[:-1], bins, weights=drawstack, histtype='stepfilled', label=sample, linewidth=1.5, color=cat_colors[sample], edgecolor=(0,0,0,0.3), alpha=1.0)
        drawstack -= bkg[key][sample][{'region': region, 'TvsQCD':sum}].values()

    error_opts = {
        'step': 'post',
        'label': 'Stat. Unc.',
        'hatch': '///',
        'facecolor': 'None',
        'edgecolor': (0, 0, 0, 0.3),
        'linewidth': 0
    }
    '''
    if region == 'sr' or 'mcr' in region:
        data[key]['MET'][{'region': region}].plot1d(ax=ax, histtype='errorbar', color='k', stack=False, label='Data')
        datastack = data[key]['MET'][{'region': region}]
    else:
        data[key]['EGamma'][{'region': region}].plot1d(ax=ax, histtype='errorbar', color='k', stack=False, label='Data')
        datastack = data[key]['EGamma'][{'region': region}]
    '''
    sig[key][sig_key1][{'region': region, 'TvsQCD':sum}].plot1d(ax=ax, histtype='step', color='c', stack=False, label=sig_key1)
    sig[key][sig_key2][{'region': region, 'TvsQCD':sum}].plot1d(ax=ax, histtype='step', color='r', stack=False, label=sig_key2)
    leg = ax.legend(ncol=3, loc='upper left', fontsize=12, title= leg_reg[region])
    ax.set_yscale('log')
    ax.set_ylim(0.01, 1000000)
    ax.set_xlabel(None)
    #rax.set_xlabel(key)
    #ds = datastack.view().value
    #ms = mcstack.view().value

    #ratio = ds / ms
    #print(ratio)

    if not key == 'template':
        lab = bkg[key][sample].axes[key].label
        rax.set_xlabel(lab)
    rax.set_ylabel('Dacta/MC')#
    rax.set_ylim(0.5, 1.5)
    if 'ut' in key:
        ax.set_xlim(300, 1000)
        rax.set_xlabel('$U_{T}$ [GeV]')

    rax.axhline(1, color='k', linestyle='--',alpha=0.5)
    rax.grid(True)

    fig.savefig('./plots'+year+'_'+whathist[histversion]+'/'+region+'_'+key+'_'+whathist[histversion]+'.png')

In [ ]:
######
## Run key loop
######
import matplotlib.pyplot as plt
import numpy as np
import mplhep as hep
from cycler import cycler
plt.style.use(hep.style.CMS)

region = 'wmcr'
stacks = hists['bkg']['sumw'].keys()
stack_map = 'tw'

keys = ['fjeta','fjphi','fjpt','jpt','jeta','jphi','met','metphi','ut','ut_bin','ut_bin2','uphi',
        'mindphirecoil','minDphirecoil','njets','njbtagL','mupt', 'mueta', 'muphi'
       ]

os.makedirs(f"plots{year}_{whathist[histversion]}", exist_ok=True)

for key in keys:
    fig, (ax, rax) = plt.subplots(
        nrows=2,
        ncols=1,
        figsize=(10,10),
        gridspec_kw={"height_ratios": (3, 1)},
        sharex=True
    )

    fig.subplots_adjust(hspace=.07)
    hep.cms.label(ax=ax, llabel='Private Work', rlabel=str(lumi[year])+' fb$^{-1}$ (13.6 TeV)')
    #ax.set_prop_cycle(cycler(color=colors[stack_map]))

    bkg  = hists['bkg']
    data = hists['data']
#sig = hists['sig']

    mcstack = None
    
    for sample in stack[stack_map]:
        #if sample == 'ST': continue
        #print(sample)
        try:
            target = bkg[key][sample][{'region': region,'TvsQCD':sum}]
            bins = target.axes.edges[0]

        except:
            print(f'In {region} sample of {sample} is Not exist')
            continue

        try:
            mcstack += bkg[key][sample][{'region': region,'TvsQCD':sum}]
            drawstack += bkg[key][sample][{'region': region,'TvsQCD':sum}].values()
        except:
            print("Start adding the stack mc & data")
            mcstack = bkg[key][sample][{'region': region,'TvsQCD':sum}]
            drawstack = bkg[key][sample][{'region': region,'TvsQCD':sum}].values()

    for sample in stack[stack_map]:
        #if sample == 'ST': continue
        if region not in bkg[key][sample].axes["region"]:
            print(f"{region} is not in {sample}'s hist axis")
            continue
        ax.hist(bins[:-1], bins, weights=drawstack, histtype='stepfilled', label=sample, linewidth=1.5, color=cat_colors[sample], edgecolor=(0,0,0,0.3), alpha=1.0)
        drawstack -= bkg[key][sample][{'region': region,'TvsQCD':sum}].values()

    error_opts = {
        'step': 'post',
        'label': 'Stat. Unc.',
        'hatch': '///',
        'facecolor': 'None',
        'edgecolor': (0, 0, 0, 0.3),
        'linewidth': 0
    }

    if region == 'sr' or 'mcr' in region:
        data[key]['MET'][{'region': region,'TvsQCD':sum}].plot1d(ax=ax, histtype='errorbar', color='k', stack=False, label='Data')
        datastack = data[key]['MET'][{'region': region,'TvsQCD':sum}]
    else:
        data[key]['EGamma'][{'region': region,'TvsQCD':sum}].plot1d(ax=ax, histtype='errorbar', color='k', stack=False, label='Data')
        datastack = data[key]['EGamma'][{'region': region,'TvsQCD':sum}]

    leg = ax.legend(ncol=3, loc='upper left', fontsize=12, title= leg_reg[region])
    ax.set_yscale('log')
    ax.set_ylim(0.01, 1000000)
    ax.set_xlabel(None)
    #rax.set_xlabel(key)
    ds = datastack.view().value
    ms = mcstack.view().value

    ratio = ds / ms
    #print(ratio)
    
    rax.errorbar(
        x=target.axes.edges[0][:-1] + np.diff(target.axes.edges[0]) / 2,
        y=ratio,
        yerr=abs(np.sqrt(datastack.view().variance) / mcstack.view().value),
        fmt="o",
        color="k",
    )
    
    rax.fill_between(
        bins,
        # append the last edge to the bins
        (1.0 - np.sqrt(mcstack.view().variance) / mcstack.view().value).tolist() + [1.0],
        (1.0 + np.sqrt(mcstack.view().variance) / mcstack.view().value).tolist() + [1.0],
        step="post",
        color="k",
        alpha=0.5,
    )
    
    if not key == 'template':
        lab = bkg[key][sample].axes[key].label
        rax.set_xlabel(lab)
    rax.set_ylabel('Dacta/MC')#
    rax.set_ylim(0.5, 1.5)
    if 'ut' in key:
        ax.set_xlim(300, 1000)
        rax.set_xlabel('$U_{T}$ [GeV]')

    rax.axhline(1, color='k', linestyle='--',alpha=0.5)
    rax.grid(True)

    fig.savefig('./plots'+year+'_'+whathist[histversion]+'/'+region+'_'+key+'_'+whathist[histversion]+'.png')

In [ ]:
######
## Run key loop
######
import matplotlib.pyplot as plt
import numpy as np
import mplhep as hep
from cycler import cycler
plt.style.use(hep.style.CMS)

region = 'wecr'
stacks = hists['bkg']['sumw'].keys()
stack_map = 'tw'

keys = ['fjeta','fjphi','fjpt','jpt','jeta','jphi','met','metphi','ut','ut_bin','ut_bin2','uphi',
        'mindphirecoil','minDphirecoil','njets','njbtagL','elept','eleeta','elephi'
       ]

os.makedirs(f"plots{year}_{whathist[histversion]}", exist_ok=True)

for key in keys:
    fig, (ax, rax) = plt.subplots(
        nrows=2,
        ncols=1,
        figsize=(10,10),
        gridspec_kw={"height_ratios": (3, 1)},
        sharex=True
    )

    fig.subplots_adjust(hspace=.07)
    hep.cms.label(ax=ax, llabel='Private Work', rlabel=str(lumi[year])+' fb$^{-1}$ (13.6 TeV)')
    #ax.set_prop_cycle(cycler(color=colors[stack_map]))

    bkg  = hists['bkg']
    data = hists['data']
#sig = hists['sig']

    mcstack = None
    
    for sample in stack[stack_map]:
        #print(sample)
        try:
            target = bkg[key][sample][{'region': region,'TvsQCD':sum}]
            bins = target.axes.edges[0]

        except:
            print(f'In {region} sample of {sample} is Not exist')
            continue

        try:
            mcstack += bkg[key][sample][{'region': region,'TvsQCD':sum}]
            drawstack += bkg[key][sample][{'region': region,'TvsQCD':sum}].values()
        except:
            print("Start adding the stack mc & data")
            mcstack = bkg[key][sample][{'region': region,'TvsQCD':sum}]
            drawstack = bkg[key][sample][{'region': region,'TvsQCD':sum}].values()

    for sample in stack[stack_map]:
        if region not in bkg[key][sample].axes["region"]:
            print(f"{region} is not in {sample}'s hist axis")
            continue
        ax.hist(bins[:-1], bins, weights=drawstack, histtype='stepfilled', label=sample, linewidth=1.5, color=cat_colors[sample], edgecolor=(0,0,0,0.3), alpha=1.0)
        drawstack -= bkg[key][sample][{'region': region,'TvsQCD':sum}].values()

    error_opts = {
        'step': 'post',
        'label': 'Stat. Unc.',
        'hatch': '///',
        'facecolor': 'None',
        'edgecolor': (0, 0, 0, 0.3),
        'linewidth': 0
    }

    if region == 'sr' or 'mcr' in region:
        data[key]['MET'][{'region': region,'TvsQCD':sum}].plot1d(ax=ax, histtype='errorbar', color='k', stack=False, label='Data')
        datastack = data[key]['MET'][{'region': region,'TvsQCD':sum}]
    else:
        data[key]['EGamma'][{'region': region,'TvsQCD':sum}].plot1d(ax=ax, histtype='errorbar', color='k', stack=False, label='Data')
        datastack = data[key]['EGamma'][{'region': region,'TvsQCD':sum}]

    leg = ax.legend(ncol=3, loc='upper left', fontsize=12, title= leg_reg[region])
    ax.set_yscale('log')
    ax.set_ylim(0.01, 1000000)
    ax.set_xlabel(None)
    #rax.set_xlabel(key)
    ds = datastack.view().value
    ms = mcstack.view().value

    ratio = ds / ms
    #print(ratio)

    rax.errorbar(
        x=target.axes.edges[0][:-1] + np.diff(target.axes.edges[0]) / 2,
        y=ratio,
        yerr=abs(np.sqrt(datastack.view().variance) / mcstack.view().value),
        fmt="o",
        color="k",
    )

    rax.fill_between(
        bins,
        # append the last edge to the bins
        (1.0 - np.sqrt(mcstack.view().variance) / mcstack.view().value).tolist() + [1.0],
        (1.0 + np.sqrt(mcstack.view().variance) / mcstack.view().value).tolist() + [1.0],
        step="post",
        color="k",
        alpha=0.5,
    )

    if not key == 'template':
        lab = bkg[key][sample].axes[key].label
        rax.set_xlabel(lab)
    rax.set_ylabel('Dacta/MC')#
    rax.set_ylim(0.5, 1.5)
    if 'ut' in key:
        ax.set_xlim(300, 1000)
        rax.set_xlabel('$U_{T}$ [GeV]')

    rax.axhline(1, color='k', linestyle='--',alpha=0.5)
    rax.grid(True)

    fig.savefig('./plots'+year+'_'+whathist[histversion]+'/'+region+'_'+key+'_'+whathist[histversion]+'.png')

In [ ]:
######
## Run key loop
######
import matplotlib.pyplot as plt
import numpy as np
import mplhep as hep
from cycler import cycler
plt.style.use(hep.style.CMS)

region = 'tmcr'

stacks = hists['bkg']['sumw'].keys()
stack_map = 'tw'

keys = ['fjeta','fjphi','fjpt','jpt','jeta','jphi','met','metphi','ut','ut_bin','ut_bin2','uphi',
        'mindphirecoil','minDphirecoil','njets','njbtagL','mupt', 'mueta', 'muphi'
       ]

os.makedirs(f"plots{year}_{whathist[histversion]}", exist_ok=True)

for key in keys:
    fig, (ax, rax) = plt.subplots(
        nrows=2,
        ncols=1,
        figsize=(10,10),
        gridspec_kw={"height_ratios": (3, 1)},
        sharex=True
    )

    fig.subplots_adjust(hspace=.07)
    hep.cms.label(ax=ax, llabel='Private Work', rlabel=str(lumi[year])+' fb$^{-1}$ (13.6 TeV)')
    #ax.set_prop_cycle(cycler(color=colors[stack_map]))

    bkg  = hists['bkg']
    data = hists['data']
#sig = hists['sig']

    mcstack = None
    
    for sample in stack[stack_map]:
        #print(sample)
        try:
            target = bkg[key][sample][{'region': region,'TvsQCD':sum}]
            bins = target.axes.edges[0]

        except:
            print(f'In {region} sample of {sample} is Not exist')
            continue

        try:
            mcstack += bkg[key][sample][{'region': region,'TvsQCD':sum}]
            drawstack += bkg[key][sample][{'region': region,'TvsQCD':sum}].values()
        except:
            print("Start adding the stack mc & data")
            mcstack = bkg[key][sample][{'region': region,'TvsQCD':sum}]
            drawstack = bkg[key][sample][{'region': region,'TvsQCD':sum}].values()

    for sample in stack[stack_map]:
        if region not in bkg[key][sample].axes["region"]:
            print(f"{region} is not in {sample}'s hist axis")
            continue
        ax.hist(bins[:-1], bins, weights=drawstack, histtype='stepfilled', label=sample, linewidth=1.5, color=cat_colors[sample], edgecolor=(0,0,0,0.3), alpha=1.0)
        drawstack -= bkg[key][sample][{'region': region,'TvsQCD':sum}].values()

    error_opts = {
        'step': 'post',
        'label': 'Stat. Unc.',
        'hatch': '///',
        'facecolor': 'None',
        'edgecolor': (0, 0, 0, 0.3),
        'linewidth': 0
    }

    if region == 'sr' or 'mcr' in region:
        data[key]['MET'][{'region': region,'TvsQCD':sum}].plot1d(ax=ax, histtype='errorbar', color='k', stack=False, label='Data')
        datastack = data[key]['MET'][{'region': region,'TvsQCD':sum}]
    else:
        data[key]['EGamma'][{'region': region,'TvsQCD':sum}].plot1d(ax=ax, histtype='errorbar', color='k', stack=False, label='Data')
        datastack = data[key]['EGamma'][{'region': region,'TvsQCD':sum}]

    leg = ax.legend(ncol=3, loc='upper left', fontsize=12, title= leg_reg[region])
    ax.set_yscale('log')
    ax.set_ylim(0.01, 1000000)
    ax.set_xlabel(None)
    #rax.set_xlabel(key)
    ds = datastack.view().value
    ms = mcstack.view().value

    ratio = ds / ms
    #print(ratio)

    rax.errorbar(
        x=target.axes.edges[0][:-1] + np.diff(target.axes.edges[0]) / 2,
        y=ratio,
        yerr=np.sqrt(datastack.view().variance) / mcstack.view().value,
        fmt="o",
        color="k",
    )

    rax.fill_between(
        bins,
        # append the last edge to the bins
        (1.0 - np.sqrt(mcstack.view().variance) / mcstack.view().value).tolist() + [1.0],
        (1.0 + np.sqrt(mcstack.view().variance) / mcstack.view().value).tolist() + [1.0],
        step="post",
        color="k",
        alpha=0.5,
    )

    if not key == 'template':
        lab = bkg[key][sample].axes[key].label
        rax.set_xlabel(lab)
    rax.set_ylabel('Dacta/MC')#
    rax.set_ylim(0.5, 1.5)
    if 'ut' in key:
        ax.set_xlim(300, 1000)
        rax.set_xlabel('$U_{T}$ [GeV]')

    rax.axhline(1, color='k', linestyle='--',alpha=0.5)
    rax.grid(True)

    fig.savefig('./plots'+year+'_'+whathist[histversion]+'/'+region+'_'+key+'_'+whathist[histversion]+'.png')

In [ ]:
######
## Run key loop
######
import matplotlib.pyplot as plt
import numpy as np
import mplhep as hep
from cycler import cycler
plt.style.use(hep.style.CMS)

region = 'tecr'
stacks = hists['bkg']['sumw'].keys()
stack_map = 'tw'

keys = ['fjeta','fjphi','fjpt','jpt','jeta','jphi','met','metphi','ut','ut_bin','ut_bin2','uphi',
        'mindphirecoil','minDphirecoil','njets','njbtagL','elept','eleeta','elephi',
       ]

os.makedirs(f"plots{year}_{whathist[histversion]}", exist_ok=True)

for key in keys:
    fig, (ax, rax) = plt.subplots(
        nrows=2,
        ncols=1,
        figsize=(10,10),
        gridspec_kw={"height_ratios": (3, 1)},
        sharex=True
    )

    fig.subplots_adjust(hspace=.07)
    hep.cms.label(ax=ax, llabel='Private Work', rlabel=str(lumi[year])+' fb$^{-1}$ (13.6 TeV)')
    #ax.set_prop_cycle(cycler(color=colors[stack_map]))

    bkg  = hists['bkg']
    data = hists['data']
#sig = hists['sig']

    mcstack = None
    
    for sample in stack[stack_map]:
        #print(sample)
        try:
            target = bkg[key][sample][{'region': region,'TvsQCD':sum}]
            bins = target.axes.edges[0]

        except:
            print(f'In {region} sample of {sample} is Not exist')
            continue

        try:
            mcstack += bkg[key][sample][{'region': region,'TvsQCD':sum}]
            drawstack += bkg[key][sample][{'region': region,'TvsQCD':sum}].values()
        except:
            print("Start adding the stack mc & data")
            mcstack = bkg[key][sample][{'region': region,'TvsQCD':sum}]
            drawstack = bkg[key][sample][{'region': region,'TvsQCD':sum}].values()

    for sample in stack[stack_map]:
        if region not in bkg[key][sample].axes["region"]:
            print(f"{region} is not in {sample}'s hist axis")
            continue
        ax.hist(bins[:-1], bins, weights=drawstack, histtype='stepfilled', label=sample, linewidth=1.5, color=cat_colors[sample], edgecolor=(0,0,0,0.3), alpha=1.0)
        drawstack -= bkg[key][sample][{'region': region,'TvsQCD':sum}].values()

    error_opts = {
        'step': 'post',
        'label': 'Stat. Unc.',
        'hatch': '///',
        'facecolor': 'None',
        'edgecolor': (0, 0, 0, 0.3),
        'linewidth': 0
    }

    if region == 'sr' or 'mcr' in region:
        data[key]['MET'][{'region': region,'TvsQCD':sum}].plot1d(ax=ax, histtype='errorbar', color='k', stack=False, label='Data')
        datastack = data[key]['MET'][{'region': region,'TvsQCD':sum}]
    else:
        data[key]['EGamma'][{'region': region,'TvsQCD':sum}].plot1d(ax=ax, histtype='errorbar', color='k', stack=False, label='Data')
        datastack = data[key]['EGamma'][{'region': region,'TvsQCD':sum}]

    leg = ax.legend(ncol=3, loc='upper left', fontsize=12, title= leg_reg[region])
    ax.set_yscale('log')
    ax.set_ylim(0.01, 1000000)
    ax.set_xlabel(None)
    #rax.set_xlabel(key)
    ds = datastack.view().value
    ms = mcstack.view().value

    ratio = ds / ms
    #print(ratio)

    rax.errorbar(
        x=target.axes.edges[0][:-1] + np.diff(target.axes.edges[0]) / 2,
        y=ratio,
        yerr=np.sqrt(datastack.view().variance) / mcstack.view().value,
        fmt="o",
        color="k",
    )

    rax.fill_between(
        bins,
        # append the last edge to the bins
        (1.0 - np.sqrt(mcstack.view().variance) / mcstack.view().value).tolist() + [1.0],
        (1.0 + np.sqrt(mcstack.view().variance) / mcstack.view().value).tolist() + [1.0],
        step="post",
        color="k",
        alpha=0.5,
    )

    if not key == 'template':
        lab = bkg[key][sample].axes[key].label
        rax.set_xlabel(lab)
    rax.set_ylabel('Dacta/MC')#
    rax.set_ylim(0.5, 1.5)
    if 'ut' in key:
        ax.set_xlim(300, 1000)
        rax.set_xlabel('$U_{T}$ [GeV]')

    rax.axhline(1, color='k', linestyle='--',alpha=0.5)
    rax.grid(True)

    fig.savefig('./plots'+year+'_'+whathist[histversion]+'/'+region+'_'+key+'_'+whathist[histversion]+'.png')

In [ ]:
######
## Run key loop
######
import matplotlib.pyplot as plt
import numpy as np
import mplhep as hep
from cycler import cycler
plt.style.use(hep.style.CMS)

region = 'zmcr'
stacks = hists['bkg']['sumw'].keys()
stack_map = 'zr'

keys = ['fjeta','fjphi','fjpt','jpt','jeta','jphi','met','metphi','ut','ut_bin','ut_bin2','uphi',
        'mindphirecoil','minDphirecoil','njets','njbtagL','dimupt','dimueta','dimuphi',
       ]

os.makedirs(f"plots{year}_{whathist[histversion]}", exist_ok=True)

for key in keys:
    fig, (ax, rax) = plt.subplots(
        nrows=2,
        ncols=1,
        figsize=(10,10),
        gridspec_kw={"height_ratios": (3, 1)},
        sharex=True
    )

    fig.subplots_adjust(hspace=.07)
    hep.cms.label(ax=ax, llabel='Private Work', rlabel=str(lumi[year])+' fb$^{-1}$ (13.6 TeV)')
    #ax.set_prop_cycle(cycler(color=colors[stack_map]))

    bkg  = hists['bkg']
    data = hists['data']
#sig = hists['sig']

    mcstack = None
    
    for sample in stack[stack_map]:
        #print(sample)
        try:
            target = bkg[key][sample][{'region': region,'TvsQCD':sum}]
            bins = target.axes.edges[0]

        except:
            print(f'In {region} sample of {sample} is Not exist')
            continue

        try:
            mcstack += bkg[key][sample][{'region': region,'TvsQCD':sum}]
            drawstack += bkg[key][sample][{'region': region,'TvsQCD':sum}].values()
        except:
            print("Start adding the stack mc & data")
            mcstack = bkg[key][sample][{'region': region,'TvsQCD':sum}]
            drawstack = bkg[key][sample][{'region': region,'TvsQCD':sum}].values()

    for sample in stack[stack_map]:
        if region not in bkg[key][sample].axes["region"]:
            print(f"{region} is not in {sample}'s hist axis")
            continue
        ax.hist(bins[:-1], bins, weights=drawstack, histtype='stepfilled', label=sample, linewidth=1.5, color=cat_colors[sample], edgecolor=(0,0,0,0.3), alpha=1.0)
        drawstack -= bkg[key][sample][{'region': region,'TvsQCD':sum}].values()

    error_opts = {
        'step': 'post',
        'label': 'Stat. Unc.',
        'hatch': '///',
        'facecolor': 'None',
        'edgecolor': (0, 0, 0, 0.3),
        'linewidth': 0
    }

    if region == 'sr' or 'mcr' in region:
        data[key]['MET'][{'region': region,'TvsQCD':sum}].plot1d(ax=ax, histtype='errorbar', color='k', stack=False, label='Data')
        datastack = data[key]['MET'][{'region': region,'TvsQCD':sum}]
    else:
        data[key]['EGamma'][{'region': region,'TvsQCD':sum}].plot1d(ax=ax, histtype='errorbar', color='k', stack=False, label='Data')
        datastack = data[key]['EGamma'][{'region': region,'TvsQCD':sum}]

    leg = ax.legend(ncol=3, loc='upper left', fontsize=12, title= leg_reg[region])
    ax.set_yscale('log')
    ax.set_ylim(0.01, 1000000)
    ax.set_xlabel(None)
    #rax.set_xlabel(key)
    ds = datastack.view().value
    ms = mcstack.view().value

    ratio = ds / ms
    #print(ratio)

    rax.errorbar(
        x=target.axes.edges[0][:-1] + np.diff(target.axes.edges[0]) / 2,
        y=ratio,
        yerr=np.sqrt(datastack.view().variance) / mcstack.view().value,
        fmt="o",
        color="k",
    )

    rax.fill_between(
        bins,
        # append the last edge to the bins
        (1.0 - np.sqrt(mcstack.view().variance) / mcstack.view().value).tolist() + [1.0],
        (1.0 + np.sqrt(mcstack.view().variance) / mcstack.view().value).tolist() + [1.0],
        step="post",
        color="k",
        alpha=0.5,
    )

    if not key == 'template':
        lab = bkg[key][sample].axes[key].label
        rax.set_xlabel(lab)
    rax.set_ylabel('Dacta/MC')#
    rax.set_ylim(0.5, 1.5)
    if 'ut' in key:
        ax.set_xlim(300, 1000)
        rax.set_xlabel('$U_{T}$ [GeV]')

    rax.axhline(1, color='k', linestyle='--',alpha=0.5)
    rax.grid(True)

    fig.savefig('./plots'+year+'_'+whathist[histversion]+'/'+region+'_'+key+'_'+whathist[histversion]+'.png')

In [ ]:
######
## Run key loop
######
import matplotlib.pyplot as plt
import numpy as np
import mplhep as hep
from cycler import cycler
plt.style.use(hep.style.CMS)

region = 'zecr'
stacks = hists['bkg']['sumw'].keys()
stack_map = 'zr'

keys = ['fjeta','fjphi','fjpt','jpt','jeta','jphi','met','metphi','ut','ut_bin','ut_bin2','uphi',
        'mindphirecoil','minDphirecoil','njets','njbtagL','TvsQCD', 'dielept','dieleeta','dielephi',
       ]

os.makedirs(f"plots{year}_{whathist[histversion]}", exist_ok=True)

for key in keys:
    fig, (ax, rax) = plt.subplots(
        nrows=2,
        ncols=1,
        figsize=(10,10),
        gridspec_kw={"height_ratios": (3, 1)},
        sharex=True
    )

    fig.subplots_adjust(hspace=.07)
    hep.cms.label(ax=ax, llabel='Private Work', rlabel=str(lumi[year])+' fb$^{-1}$ (13.6 TeV)')
    #ax.set_prop_cycle(cycler(color=colors[stack_map]))

    bkg  = hists['bkg']
    data = hists['data']
#sig = hists['sig']

    mcstack = None
    
    for sample in stack[stack_map]:
        #print(sample)
        try:
            target = bkg[key][sample][{'region': region,'TvsQCD':sum}]
            bins = target.axes.edges[0]

        except:
            print(f'In {region} sample of {sample} is Not exist')
            continue

        try:
            mcstack += bkg[key][sample][{'region': region,'TvsQCD':sum}]
            drawstack += bkg[key][sample][{'region': region,'TvsQCD':sum}].values()
        except:
            print("Start adding the stack mc & data")
            mcstack = bkg[key][sample][{'region': region,'TvsQCD':sum}]
            drawstack = bkg[key][sample][{'region': region,'TvsQCD':sum}].values()

    for sample in stack[stack_map]:
        if region not in bkg[key][sample].axes["region"]:
            print(f"{region} is not in {sample}'s hist axis")
            continue
        ax.hist(bins[:-1], bins, weights=drawstack, histtype='stepfilled', label=sample, linewidth=1.5, color=cat_colors[sample], edgecolor=(0,0,0,0.3), alpha=1.0)
        drawstack -= bkg[key][sample][{'region': region,'TvsQCD':sum}].values()

    error_opts = {
        'step': 'post',
        'label': 'Stat. Unc.',
        'hatch': '///',
        'facecolor': 'None',
        'edgecolor': (0, 0, 0, 0.3),
        'linewidth': 0
    }

    if region == 'sr' or 'mcr' in region:
        data[key]['MET'][{'region': region,'TvsQCD':sum}].plot1d(ax=ax, histtype='errorbar', color='k', stack=False, label='Data')
        datastack = data[key]['MET'][{'region': region,'TvsQCD':sum}]
    else:
        data[key]['EGamma'][{'region': region,'TvsQCD':sum}].plot1d(ax=ax, histtype='errorbar', color='k', stack=False, label='Data')
        datastack = data[key]['EGamma'][{'region': region,'TvsQCD':sum}]

    leg = ax.legend(ncol=3, loc='upper left', fontsize=12, title= leg_reg[region])
    ax.set_yscale('log')
    ax.set_ylim(0.01, 1000000)
    ax.set_xlabel(None)
    #rax.set_xlabel(key)
    ds = datastack.view().value
    ms = mcstack.view().value

    ratio = ds / ms
    #print(ratio)

    rax.errorbar(
        x=target.axes.edges[0][:-1] + np.diff(target.axes.edges[0]) / 2,
        y=ratio,
        yerr=np.sqrt(datastack.view().variance) / mcstack.view().value,
        fmt="o",
        color="k",
    )

    rax.fill_between(
        bins,
        # append the last edge to the bins
        (1.0 - np.sqrt(mcstack.view().variance) / mcstack.view().value).tolist() + [1.0],
        (1.0 + np.sqrt(mcstack.view().variance) / mcstack.view().value).tolist() + [1.0],
        step="post",
        color="k",
        alpha=0.5,
    )

    if not key == 'template':
        lab = bkg[key][sample].axes[key].label
        rax.set_xlabel(lab)
    rax.set_ylabel('Dacta/MC')#
    rax.set_ylim(0.5, 1.5)
    if 'ut' in key:
        ax.set_xlim(300, 1000)
        rax.set_xlabel('$U_{T}$ [GeV]')

    rax.axhline(1, color='k', linestyle='--',alpha=0.5)
    rax.grid(True)

    fig.savefig('./plots'+year+'_'+whathist[histversion]+'/'+region+'_'+key+'_'+whathist[histversion]+'.png')

In [ ]:
######
## Run key loop
######
import matplotlib.pyplot as plt
import numpy as np
import mplhep as hep
from cycler import cycler
plt.style.use(hep.style.CMS)

region = 'gcr'
stacks = hists['bkg']['sumw'].keys()
stack_map = 'gr'

keys = ['fjeta','fjphi','fjpt','jpt','jeta','jphi','met','metphi','ut','ut_bin','ut_bin2','uphi',
        'mindphirecoil','minDphirecoil','njets','njbtagL','TvsQCD','phopt','phoeta','phophi',
       ]

os.makedirs(f"plots{year}_{whathist[histversion]}", exist_ok=True)

for key in keys:
    fig, (ax, rax) = plt.subplots(
        nrows=2,
        ncols=1,
        figsize=(10,10),
        gridspec_kw={"height_ratios": (3, 1)},
        sharex=True
    )

    fig.subplots_adjust(hspace=.07)
    hep.cms.label(ax=ax, llabel='Private Work', rlabel=str(lumi[year])+' fb$^{-1}$ (13.6 TeV)')
    #ax.set_prop_cycle(cycler(color=colors[stack_map]))

    bkg  = hists['bkg']
    data = hists['data']
#sig = hists['sig']

    mcstack = None
    
    for sample in stack[stack_map]:
        #print(sample)
        try:
            target = bkg[key][sample][{'region': region,'TvsQCD':sum}]
            bins = target.axes.edges[0]

        except:
            print(f'In {region} sample of {sample} is Not exist')
            continue

        try:
            mcstack += bkg[key][sample][{'region': region,'TvsQCD':sum}]
            drawstack += bkg[key][sample][{'region': region,'TvsQCD':sum}].values()
        except:
            print("Start adding the stack mc & data")
            mcstack = bkg[key][sample][{'region': region,'TvsQCD':sum}]
            drawstack = bkg[key][sample][{'region': region,'TvsQCD':sum}].values()

    for sample in stack[stack_map]:
        if region not in bkg[key][sample].axes["region"]:
            print(f"{region} is not in {sample}'s hist axis")
            continue
        ax.hist(bins[:-1], bins, weights=drawstack, histtype='stepfilled', label=sample, linewidth=1.5, color=cat_colors[sample], edgecolor=(0,0,0,0.3), alpha=1.0)
        drawstack -= bkg[key][sample][{'region': region,'TvsQCD':sum}].values()

    error_opts = {
        'step': 'post',
        'label': 'Stat. Unc.',
        'hatch': '///',
        'facecolor': 'None',
        'edgecolor': (0, 0, 0, 0.3),
        'linewidth': 0
    }

    if region == 'sr' or 'mcr' in region:
        data[key]['MET'][{'region': region,'TvsQCD':sum}].plot1d(ax=ax, histtype='errorbar', color='k', stack=False, label='Data')
        datastack = data[key]['MET'][{'region': region,'TvsQCD':sum}]
    else:
        data[key]['EGamma'][{'region': region,'TvsQCD':sum}].plot1d(ax=ax, histtype='errorbar', color='k', stack=False, label='Data')
        datastack = data[key]['EGamma'][{'region': region,'TvsQCD':sum}]

    leg = ax.legend(ncol=3, loc='upper left', fontsize=12, title= leg_reg[region])
    ax.set_yscale('log')
    ax.set_ylim(0.01, 1000000)
    ax.set_xlabel(None)
    #rax.set_xlabel(key)
    ds = datastack.view().value
    ms = mcstack.view().value

    ratio = ds / ms
    #print(ratio)

    rax.errorbar(
        x=target.axes.edges[0][:-1] + np.diff(target.axes.edges[0]) / 2,
        y=ratio,
        yerr=np.sqrt(datastack.view().variance) / mcstack.view().value,
        fmt="o",
        color="k",
    )

    rax.fill_between(
        bins,
        # append the last edge to the bins
        (1.0 - np.sqrt(mcstack.view().variance) / mcstack.view().value).tolist() + [1.0],
        (1.0 + np.sqrt(mcstack.view().variance) / mcstack.view().value).tolist() + [1.0],
        step="post",
        color="k",
        alpha=0.5,
    )

    if not key == 'template':
        lab = bkg[key][sample].axes[key].label
        rax.set_xlabel(lab)
    rax.set_ylabel('Dacta/MC')#
    rax.set_ylim(0.5, 1.5)
    if 'ut' in key:
        ax.set_xlim(300, 1000)
        rax.set_xlabel('$U_{T}$ [GeV]')

    rax.axhline(1, color='k', linestyle='--',alpha=0.5)
    rax.grid(True)

    fig.savefig('./plots'+year+'_'+whathist[histversion]+'/'+region+'_'+key+'_'+whathist[histversion]+'.png')

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import mplhep as hep
from cycler import cycler
plt.style.use(hep.style.CMS)

#year = '2022pre'
region = 'sr'
key = 'genWpt'
stacks = hists['bkg']['sumw'].keys()
stack_map = 'sr'

regions = ['sr', 'wmcr', 'wecr', 'tmcr', 'gcr', 'tecr', 'zecr', 'zmcr']
leg_reg = {
    'sr': 'SR', 'wmcr': r'W($\mu$) CR', 'wecr': r'W(e) CR', 'tmcr': r'Top($\mu$) CR', 'gcr': 'G CR',
    'tecr': r'Top(e) CR', 'zecr':r'Z(ee) CR', 'zmcr': r'Z($\mu\mu$) CR'
}

fig, (ax, rax) = plt.subplots(
    nrows=2,
    ncols=1,
    figsize=(10,10),
    gridspec_kw={"height_ratios": (3, 1)},
    sharex=True
)

fig.subplots_adjust(hspace=.07)
hep.cms.label(ax=ax, llabel='Private Work', rlabel=str(lumi[year])+' fb$^{-1}$ (13 TeV)')
ax.set_prop_cycle(cycler(color=colors[stack_map]))

bkg  = hists['bkg']


mcstack = None
for sample in stack[stack_map]:
    #print(sample)
    if not 'W ' in sample: continue
    print(sample)
    try:
        target = bkg[key][sample][{'region': region}]
        bins = target.axes.edges[0]

    except:
        print(f'In {region} sample of {sample} is Not exist')
        continue

    try:
        mcstack += bkg[key][sample][{'region': region}]
        drawstack += bkg[key][sample][{'region': region}].values()
    except:
        print("Start adding the stack mc & data")
        mcstack = bkg[key][sample][{'region': region}]
        drawstack = bkg[key][sample][{'region': region}].values()

for sample in stack[stack_map]:
    if region not in bkg[key][sample].axes["region"]:
        print(f"{region} is not in {sample}'s hist axis")
        continue
    ax.hist(bins[:-1], bins, weights=drawstack, histtype='stepfilled', label=sample, linewidth=1.5, edgecolor=(0,0,0,0.3), alpha=1.0)
    drawstack -= bkg[key][sample][{'region': region}].values()
    
error_opts = {
    'step': 'post',
    'label': 'Stat. Unc.',
    'hatch': '///',
    'facecolor': 'None',
    'edgecolor': (0, 0, 0, 0.3),
    'linewidth': 0
}


leg = ax.legend(ncol=3, loc='upper left', fontsize=12, title= leg_reg[region])
ax.set_yscale('log')
ax.set_ylim(0.01, 1000000)
ax.set_xlabel(None)
ax.axvline(120, color='red', linestyle='--', linewidth=1.5)

if not key == 'template':
    lab = bkg[key][sample].axes[key].label
    rax.set_xlabel(lab)
rax.set_ylabel('Dacta/MC')#
rax.set_ylim(0.5, 1.5)

rax.axhline(1, color='k', linestyle='--',alpha=0.5)
rax.grid(True)

fig.savefig('./plots'+year+'_'+whathist[histversion]+'/'+region+'_'+key+'_'+whathist[histversion]+'.png')


In [ ]:
######
## Run TvsQCD loop: SR 
######
import matplotlib.pyplot as plt
import numpy as np
import mplhep as hep
from cycler import cycler
plt.style.use(hep.style.CMS)

region = 'sr'
stacks = hists['bkg']['sumw'].keys()
stack_map = 'sr'

keys = ['TvsQCD'
       ]
sig_key1 = 'sig_Mphi-2000_Mchi-150'
sig_key2 = 'sig_Mphi-200_Mchi-150'
os.makedirs(f"plots{year}_{whathist[histversion]}", exist_ok=True)

for key in keys:
    fig, (ax, rax) = plt.subplots(
        nrows=2,
        ncols=1,
        figsize=(10,10),
        gridspec_kw={"height_ratios": (3, 1)},
        sharex=True
    )

    fig.subplots_adjust(hspace=.07)
    hep.cms.label(ax=ax, llabel='Private Work', rlabel=str(lumi[year])+' fb$^{-1}$ (13.6 TeV)')
    #ax.set_prop_cycle(cycler(color=colors[stack_map]))

    bkg  = hists['bkg']
    data = hists['data']
    sig  = hists['sig']

    mcstack = None
    
    for sample in stack[stack_map]:
        #if 'Z ($\nu\nu$)' in sample: continue
        #print(sample)
        try:
            target = bkg[key][sample][{'region': region}]
            bins = target.axes.edges[0]

        except:
            print(f'In {region} sample of {sample} is Not exist')
            continue

        try:
            mcstack += bkg[key][sample][{'region': region}]
            drawstack += bkg[key][sample][{'region': region}].values()
        except:
            print("Start adding the stack mc & data")
            mcstack = bkg[key][sample][{'region': region}]
            drawstack = bkg[key][sample][{'region': region}].values()

    for sample in stack[stack_map]:
        if region not in bkg[key][sample].axes["region"]:
            print(f"{region} is not in {sample}'s hist axis")
            continue
        ax.hist(bins[:-1], bins, weights=drawstack, histtype='stepfilled', label=sample, linewidth=1.5, color=cat_colors[sample], edgecolor=(0,0,0,0.3), alpha=1.0)
        drawstack -= bkg[key][sample][{'region': region}].values()

    error_opts = {
        'step': 'post',
        'label': 'Stat. Unc.',
        'hatch': '///',
        'facecolor': 'None',
        'edgecolor': (0, 0, 0, 0.3),
        'linewidth': 0
    }
    '''
    if region == 'sr' or 'mcr' in region:
        data[key]['MET'][{'region': region}].plot1d(ax=ax, histtype='errorbar', color='k', stack=False, label='Data')
        datastack = data[key]['MET'][{'region': region}]
    else:
        data[key]['EGamma'][{'region': region}].plot1d(ax=ax, histtype='errorbar', color='k', stack=False, label='Data')
        datastack = data[key]['EGamma'][{'region': region}]
    '''
    sig[key][sig_key1][{'region': region}].plot1d(ax=ax, histtype='step', color='c', stack=False, label=sig_key1)
    sig[key][sig_key2][{'region': region}].plot1d(ax=ax, histtype='step', color='r', stack=False, label=sig_key2)
    leg = ax.legend(ncol=3, loc='upper left', fontsize=12, title= leg_reg[region])
    ax.set_yscale('log')
    ax.set_ylim(0.01, 1000000)
    ax.set_xlabel(None)
    #rax.set_xlabel(key)
    #ds = datastack.view().value
    #ms = mcstack.view().value

    #ratio = ds / ms
    #print(ratio)

    if not key == 'template':
        lab = bkg[key][sample].axes[key].label
        rax.set_xlabel(lab)
    rax.set_ylabel('Dacta/MC')#
    rax.set_ylim(0.5, 1.5)
    if 'ut' in key:
        ax.set_xlim(300, 1000)
        rax.set_xlabel('$U_{T}$ [GeV]')

    rax.axhline(1, color='k', linestyle='--',alpha=0.5)
    rax.grid(True)

    fig.savefig('./plots'+year+'_'+whathist[histversion]+'/'+region+'_'+key+'_'+whathist[histversion]+'.png')

In [ ]:
######
## Run Recoil binning playground 
######
import matplotlib.pyplot as plt
import numpy as np
import mplhep as hep
from cycler import cycler
plt.style.use(hep.style.CMS)

region = 'tecr'
stacks = hists['bkg']['sumw'].keys()
stack_map = 'tw'
new_bin = [250, 350, 450, 550, 650, 1000]
keys = ['ut'
       ]
sig_key1 = 'sig_Mphi-2000_Mchi-150'
sig_key2 = 'sig_Mphi-200_Mchi-150'
os.makedirs(f"plots{year}_{whathist[histversion]}", exist_ok=True)

for key in keys:
    fig, (ax, rax) = plt.subplots(
        nrows=2,
        ncols=1,
        figsize=(10,10),
        gridspec_kw={"height_ratios": (3, 1)},
        sharex=True
    )

    fig.subplots_adjust(hspace=.07)
    hep.cms.label(ax=ax, llabel='Private Work', rlabel=str(lumi[year])+' fb$^{-1}$ (13.6 TeV)')
    #ax.set_prop_cycle(cycler(color=colors[stack_map]))

    bkg  = hists['bkg']
    data = hists['data']
    sig  = hists['sig']

    mcstack = None
    
    for sample in stack[stack_map]:
        try:
            #target = bkg[key][sample][{'region': region,'TvsQCD':sum}]
            target = bkg[key][sample][{'region': region,'TvsQCD': sum,'ut': hist.rebin(edges=new_bin)}]
            bins = target.axes.edges[0]

        except:
            print(f'In {region} sample of {sample} is Not exist')
            continue

        try:
            mcstack += bkg[key][sample][{'region': region,'TvsQCD':sum,'ut': hist.rebin(edges=new_bin)}]
            drawstack += bkg[key][sample][{'region': region,'TvsQCD':sum,'ut': hist.rebin(edges=new_bin)}].values()
        except:
            print("Start adding the stack mc & data")
            mcstack = bkg[key][sample][{'region': region,'TvsQCD':sum,'ut': hist.rebin(edges=new_bin)}]
            drawstack = bkg[key][sample][{'region': region,'TvsQCD':sum,'ut': hist.rebin(edges=new_bin)}].values()
    print(target.axes.edges[0])
    for sample in stack[stack_map]:
        if region not in bkg[key][sample].axes["region"]:
            print(f"{region} is not in {sample}'s hist axis")
            continue
        ax.hist(bins[:-1], bins, weights=drawstack, histtype='stepfilled', label=sample, linewidth=1.5, color=cat_colors[sample], edgecolor=(0,0,0,0.3), alpha=1.0)
        drawstack -= bkg[key][sample][{'region': region,'TvsQCD':sum,'ut': hist.rebin(edges=new_bin)}].values()

    error_opts = {
        'step': 'post',
        'label': 'Stat. Unc.',
        'hatch': '///',
        'facecolor': 'None',
        'edgecolor': (0, 0, 0, 0.3),
        'linewidth': 0
    }
    if region != 'sr':
        if region == 'sr' or 'mcr' in region:
            data[key]['MET'][{'region': region,'TvsQCD':sum,'ut': hist.rebin(edges=new_bin)}].plot1d(ax=ax, histtype='errorbar', color='k', stack=False, label='Data')
            datastack = data[key]['MET'][{'region': region,'TvsQCD':sum,'ut': hist.rebin(edges=new_bin)}]
        else:
            data[key]['EGamma'][{'region': region,'TvsQCD':sum,'ut': hist.rebin(edges=new_bin)}].plot1d(ax=ax, histtype='errorbar', color='k', stack=False, label='Data')
            datastack = data[key]['EGamma'][{'region': region,'TvsQCD':sum,'ut': hist.rebin(edges=new_bin)}]
    
    sig[key][sig_key1][{'region': region,'TvsQCD':sum,'ut': hist.rebin(edges=new_bin)}].plot1d(ax=ax, histtype='step', color='c', stack=False, label=sig_key1)
    sig[key][sig_key2][{'region': region,'TvsQCD':sum,'ut': hist.rebin(edges=new_bin)}].plot1d(ax=ax, histtype='step', color='r', stack=False, label=sig_key2)
    leg = ax.legend(ncol=3, loc='upper left', fontsize=12, title= leg_reg[region])
    ax.set_yscale('log')
    ax.set_ylim(0.01, 1000000)
    ax.set_xlabel(None)
    rax.set_xlabel(key)

    if region != 'sr':    
        ds = datastack.view().value
        ms = mcstack.view().value
        ratio = ds / ms

        rax.errorbar(
            x=target.axes.edges[0][:-1] + np.diff(target.axes.edges[0]) / 2,
            y=ratio,
            yerr=np.sqrt(datastack.view().variance) / mcstack.view().value,
            fmt="o",
            color="k",
        )

        rax.fill_between(
            bins,
            # append the last edge to the bins
            (1.0 - np.sqrt(mcstack.view().variance) / mcstack.view().value).tolist() + [1.0],
            (1.0 + np.sqrt(mcstack.view().variance) / mcstack.view().value).tolist() + [1.0],
            step="post",
            color="k",
            alpha=0.5,
        )
        
    if not key == 'template':
        lab = bkg[key][sample].axes[key].label
        rax.set_xlabel(lab)
    rax.set_ylabel('Dacta/MC')#
    rax.set_ylim(0.5, 1.5)
    if 'ut' in key:
        ax.set_xlim(300, 1000)
        rax.set_xlabel('$U_{T}$ [GeV]')

    rax.axhline(1, color='k', linestyle='--',alpha=0.5)
    rax.grid(True)

    #fig.savefig('./plots'+year+'_'+whathist[histversion]+'/'+region+'_'+key+'_'+whathist[histversion]+'.png')

In [ ]:
print(hist.__version__)

In [ ]:
######
## Run TvsQCD loop: CR 
######
import matplotlib.pyplot as plt
import numpy as np
import mplhep as hep
from cycler import cycler
plt.style.use(hep.style.CMS)

regions = ['wmcr', 'wecr', 'tmcr', 'gcr', 'tecr', 'zecr', 'zmcr']
stacks = hists['bkg']['sumw'].keys()
stack_map = 'sr'

keys = ['TvsQCD'
       ]
key = 'TvsQCD'
sig_key1 = 'sig_Mphi-2000_Mchi-150'
sig_key2 = 'sig_Mphi-200_Mchi-150'
os.makedirs(f"plots{year}_{whathist[histversion]}", exist_ok=True)

for reg in regions:
    fig, (ax, rax) = plt.subplots(
        nrows=2,
        ncols=1,
        figsize=(10,10),
        gridspec_kw={"height_ratios": (3, 1)},
        sharex=True
    )

    fig.subplots_adjust(hspace=.07)
    hep.cms.label(ax=ax, llabel='Private Work', rlabel=str(lumi[year])+' fb$^{-1}$ (13.6 TeV)')
    #ax.set_prop_cycle(cycler(color=colors[stack_map]))

    bkg  = hists['bkg']
    data = hists['data']
    sig  = hists['sig']

    mcstack = None
    
    for sample in stack[stack_map]:
        #if 'Z ($\nu\nu$)' in sample: continue
        #print(sample)
        try:
            target = bkg[key][sample][{'region': reg}]
            bins = target.axes.edges[0]

        except:
            print(f'In {reg} sample of {sample} is Not exist')
            continue

        try:
            mcstack += bkg[key][sample][{'region': reg}]
            drawstack += bkg[key][sample][{'region': reg}].values()
        except:
            print("Start adding the stack mc & data")
            mcstack = bkg[key][sample][{'region': reg}]
            drawstack = bkg[key][sample][{'region': reg}].values()

    for sample in stack[stack_map]:
        if reg not in bkg[key][sample].axes["region"]:
            print(f"{reg} is not in {sample}'s hist axis")
            continue
        ax.hist(bins[:-1], bins, weights=drawstack, histtype='stepfilled', label=sample, linewidth=1.5, color=cat_colors[sample], edgecolor=(0,0,0,0.3), alpha=1.0)
        drawstack -= bkg[key][sample][{'region': reg}].values()

    error_opts = {
        'step': 'post',
        'label': 'Stat. Unc.',
        'hatch': '///',
        'facecolor': 'None',
        'edgecolor': (0, 0, 0, 0.3),
        'linewidth': 0
    }
    '''
    if region == 'sr' or 'mcr' in region:
        data[key]['MET'][{'region': region}].plot1d(ax=ax, histtype='errorbar', color='k', stack=False, label='Data')
        datastack = data[key]['MET'][{'region': region}]
    else:
        data[key]['EGamma'][{'region': region}].plot1d(ax=ax, histtype='errorbar', color='k', stack=False, label='Data')
        datastack = data[key]['EGamma'][{'region': region}]
    '''
    sig[key][sig_key1][{'region': reg}].plot1d(ax=ax, histtype='step', color='c', stack=False, label=sig_key1)
    sig[key][sig_key2][{'region': reg}].plot1d(ax=ax, histtype='step', color='r', stack=False, label=sig_key2)
    leg = ax.legend(ncol=3, loc='upper left', fontsize=12, title= leg_reg[reg])
    ax.set_yscale('log')
    ax.set_ylim(0.01, 1000000)
    ax.set_xlabel(None)
    #rax.set_xlabel(key)
    #ds = datastack.view().value
    #ms = mcstack.view().value

    #ratio = ds / ms
    #print(ratio)

    if not key == 'template':
        lab = bkg[key][sample].axes[key].label
        rax.set_xlabel(lab)
    rax.set_ylabel('Dacta/MC')#
    rax.set_ylim(0.5, 1.5)
    if 'ut' in key:
        ax.set_xlim(300, 1000)
        rax.set_xlabel('$U_{T}$ [GeV]')

    rax.axhline(1, color='k', linestyle='--',alpha=0.5)
    rax.grid(True)

    fig.savefig('./plots'+year+'_'+whathist[histversion]+'/'+reg+'_'+key+'_'+whathist[histversion]+'.png')